In [6]:
import pandas as pd
import numpy as np

base = "/home/darshani/lightkurve-env/space-debris-detector/data"

tle = pd.read_parquet(f"{base}/cleaned/tle_cleaned.parquet")
satcat = pd.read_parquet(f"{base}/cleaned/satcat_cleaned.parquet")
decay = pd.read_parquet(f"{base}/cleaned/decay_cleaned.parquet")
conj = pd.read_parquet(f"{base}/cleaned/conjunctions_cleaned.parquet")
discos = pd.read_parquet(f"{base}/cleaned/discos_cleaned.parquet")
sgp4 = pd.read_parquet(f"{base}/sgp4_positions.parquet")

print(tle.shape, satcat.shape, decay.shape, conj.shape, discos.shape, sgp4.shape)

(66727, 40) (68151, 24) (134686, 13) (4013, 16) (89749, 31) (37892, 12)


In [7]:
for name, df in [('TLE', tle), ('SATCAT', satcat), ('DECAY', decay), ('CONJ', conj), ('DISCOS', discos), ('SGP4', sgp4)]:
    print(f"\n{name}:", df.columns.tolist())


TLE: ['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'OBJECT_NAME', 'OBJECT_ID', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID', 'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD', 'APOAPSIS', 'PERIAPSIS', 'OBJECT_TYPE', 'RCS_SIZE', 'COUNTRY_CODE', 'LAUNCH_DATE', 'SITE', 'DECAY_DATE', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2']

SATCAT: ['INTLDES', 'NORAD_CAT_ID', 'OBJECT_TYPE', 'SATNAME', 'COUNTRY', 'LAUNCH', 'SITE', 'DECAY', 'PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE', 'COMMENT', 'COMMENTCODE', 'RCSVALUE', 'RCS_SIZE', 'FILE', 'LAUNCH_YEAR', 'LAUNCH_NUM', 'LAUNCH_PIECE', 'CURRENT', 'OBJECT_NAME', 'OBJECT_ID', 'OBJECT_NUMBER']

DECAY: ['NORAD_CAT_ID', 'OBJECT_NUMBER', 'OBJECT_NAME', 'INTLDES', 'OBJECT_ID', 'RCS', 'RCS_SI

In [8]:
for df in [tle, satcat, decay, sgp4]:
    df['NORAD_CAT_ID'] = pd.to_numeric(df['NORAD_CAT_ID'], errors='coerce').astype('Int64')

# discos uses attr_satno instead of NORAD_CAT_ID
discos = discos.rename(columns={'attr_satno': 'NORAD_CAT_ID'})
discos['NORAD_CAT_ID'] = pd.to_numeric(discos['NORAD_CAT_ID'], errors='coerce').astype('Int64')

# conj has two objects per row
conj['SAT_1_ID'] = pd.to_numeric(conj['SAT_1_ID'], errors='coerce').astype('Int64')
conj['SAT_2_ID'] = pd.to_numeric(conj['SAT_2_ID'], errors='coerce').astype('Int64')

print("ids fixed!!")

ids fixed!!


In [9]:
decay = decay.sort_values('MSG_EPOCH').groupby('NORAD_CAT_ID', as_index=False).last()
print("decay:", decay.shape)

decay: (35875, 13)


In [10]:
conj['PC'] = pd.to_numeric(conj['PC'], errors='coerce')
conj['MIN_RNG'] = pd.to_numeric(conj['MIN_RNG'], errors='coerce')

sat1 = conj[['SAT_1_ID', 'PC', 'MIN_RNG']].rename(columns={'SAT_1_ID': 'NORAD_CAT_ID'})
sat2 = conj[['SAT_2_ID', 'PC', 'MIN_RNG']].rename(columns={'SAT_2_ID': 'NORAD_CAT_ID'})
conj_agg = pd.concat([sat1, sat2]).groupby('NORAD_CAT_ID').agg(
    conj_count=('PC', 'count'),
    max_pc=('PC', 'max'),
    mean_pc=('PC', 'mean'),
    min_miss_dist=('MIN_RNG', 'min')
).reset_index()

print("conj:", conj_agg.shape)

conj: (1041, 5)


In [11]:
merged = tle.copy()
print("start:", merged.shape)

merged = merged.merge(satcat, on='NORAD_CAT_ID', how='left', suffixes=('', '_satcat'))
print("after satcat:", merged.shape)

merged = merged.merge(decay, on='NORAD_CAT_ID', how='left', suffixes=('', '_decay'))
print("after decay:", merged.shape)

merged = merged.merge(conj_agg, on='NORAD_CAT_ID', how='left')
print("after conj:", merged.shape)

merged = merged.merge(discos, on='NORAD_CAT_ID', how='left', suffixes=('', '_discos'))
print("after discos:", merged.shape)

merged = merged.merge(sgp4, on='NORAD_CAT_ID', how='left')
print("after sgp4:", merged.shape)

start: (66727, 40)
after satcat: (66727, 63)
after decay: (66727, 75)
after conj: (66727, 79)
after discos: (66727, 109)
after sgp4: (66727, 120)


In [18]:
merged.to_parquet(f"{base}/merged/all_merged.parquet", index=False)
print("done!!", merged.shape)
merged.head()

done!! (66727, 120)


,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,...,rx_km,ry_km,rz_km,vx_km_s,vy_km_s,vz_km_s,tle_age_days,altitude_km,speed_km_s,regime
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:37:47,18 SPCS,EXPLORER 1,1958-001A,EARTH,TEME,UTC,SGP4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-21T03:14:42,18 SPCS,VANGUARD 1,1958-002B,EARTH,TEME,UTC,SGP4,...,-8210.998356,-3990.498411,-1923.840662,2.574536,-4.548402,3.446308,1.0,2958.830356,6.260450,MEO
2,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,SPUTNIK 3,1958-004B,EARTH,TEME,UTC,SGP4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:20,18 SPCS,EXPLORER 4,1958-005A,EARTH,TEME,UTC,SGP4,...,-3519.404089,3015.901726,-4733.148996,-6.414072,-3.952547,2.200901,24277.0,253.542836,7.849007,VLEO
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-21T17:23:35,18 SPCS,VANGUARD 2,1959-001A,EARTH,TEME,UTC,SGP4,...,-1388.811012,7840.407708,3589.239802,-5.787650,-1.342493,2.607252,0.0,2363.038666,6.488216,MEO


In [19]:
merged = pd.read_parquet("/home/darshani/lightkurve-env/space-debris-detector/data/merged/all_merged.parquet")

# find duplicate columns (the _satcat, _decay, _discos suffixed ones)
dup_cols = [c for c in merged.columns if c.endswith('_satcat') or c.endswith('_decay') or c.endswith('_discos')]
print("duplicate cols to drop:", dup_cols)
print("shape before:", merged.shape)

duplicate cols to drop: ['OBJECT_TYPE_satcat', 'SITE_satcat', 'PERIOD_satcat', 'INCLINATION_satcat', 'COMMENT_satcat', 'RCS_SIZE_satcat', 'FILE_satcat', 'OBJECT_NAME_satcat', 'OBJECT_ID_satcat', 'OBJECT_NUMBER_decay', 'OBJECT_NAME_decay', 'INTLDES_decay', 'OBJECT_ID_decay', 'RCS_SIZE_decay', 'COUNTRY_decay']
shape before: (66727, 120)


In [20]:
dup_cols = ['OBJECT_TYPE_satcat', 'SITE_satcat', 'PERIOD_satcat', 'INCLINATION_satcat', 
            'COMMENT_satcat', 'RCS_SIZE_satcat', 'FILE_satcat', 'OBJECT_NAME_satcat', 
            'OBJECT_ID_satcat', 'OBJECT_NUMBER_decay', 'OBJECT_NAME_decay', 'INTLDES_decay', 
            'OBJECT_ID_decay', 'RCS_SIZE_decay', 'COUNTRY_decay']

merged = merged.drop(columns=dup_cols)
print("shape after dropping dupes:", merged.shape)

shape after dropping dupes: (66727, 105)


In [21]:
merged.to_parquet(f"{base}/merged/all_cleaned_merged.parquet", index=False)
print("saved!!", merged.shape)
merged.head()

saved!! (66727, 105)


,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,...,rx_km,ry_km,rz_km,vx_km_s,vy_km_s,vz_km_s,tle_age_days,altitude_km,speed_km_s,regime
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:37:47,18 SPCS,EXPLORER 1,1958-001A,EARTH,TEME,UTC,SGP4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-21T03:14:42,18 SPCS,VANGUARD 1,1958-002B,EARTH,TEME,UTC,SGP4,...,-8210.998356,-3990.498411,-1923.840662,2.574536,-4.548402,3.446308,1.0,2958.830356,6.260450,MEO
2,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,SPUTNIK 3,1958-004B,EARTH,TEME,UTC,SGP4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:20,18 SPCS,EXPLORER 4,1958-005A,EARTH,TEME,UTC,SGP4,...,-3519.404089,3015.901726,-4733.148996,-6.414072,-3.952547,2.200901,24277.0,253.542836,7.849007,VLEO
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-21T17:23:35,18 SPCS,VANGUARD 2,1959-001A,EARTH,TEME,UTC,SGP4,...,-1388.811012,7840.407708,3589.239802,-5.787650,-1.342493,2.607252,0.0,2363.038666,6.488216,MEO


In [22]:
print(merged.columns.tolist())

['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'OBJECT_NAME', 'OBJECT_ID', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID', 'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD', 'APOAPSIS', 'PERIAPSIS', 'OBJECT_TYPE', 'RCS_SIZE', 'COUNTRY_CODE', 'LAUNCH_DATE', 'SITE', 'DECAY_DATE', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2', 'INTLDES', 'SATNAME', 'COUNTRY', 'LAUNCH', 'DECAY', 'APOGEE', 'PERIGEE', 'COMMENTCODE', 'RCSVALUE', 'LAUNCH_YEAR', 'LAUNCH_NUM', 'LAUNCH_PIECE', 'CURRENT', 'OBJECT_NUMBER', 'RCS', 'MSG_EPOCH', 'DECAY_EPOCH', 'SOURCE', 'MSG_TYPE', 'PRECEDENCE', 'conj_count', 'max_pc', 'mean_pc', 'min_miss_dist', 'type', 'id', 'attr_cosparId', 'attr_vimpelId', 'attr_name', 'attr_objectClass', 'attr_mass', 'attr_shape', 'at

In [23]:
# not useful for the model at all
drop_cols = [
    'CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR',
    'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY',
    'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'ELEMENT_SET_NO',
    'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2',
    'SITE', 'INTLDES', 'SATNAME', 'COMMENTCODE',
    'OBJECT_NUMBER', 'SOURCE', 'MSG_TYPE',
    'PRECEDENCE', 'type', 'id', 'attr_cosparId', 'attr_vimpelId',
    'attr_name', 'attr_mission', 'rel_launch', 'rel_reentry',
    'rel_initialOrbits', 'rel_destinationOrbits', 'rel_states',
    'rel_operators', 'rel_tags', 'rel_constellations',
    'OBJECT_ID', 'MSG_EPOCH', 'error', 'RA_OF_ASC_NODE',
    'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'MEAN_MOTION_DDOT',
    'REV_AT_EPOCH', 'LAUNCH_NUM', 'LAUNCH_PIECE', 'LAUNCH',
    'COUNTRY_CODE', 'COUNTRY'
]

merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns])
print("final shape:", merged.shape)
merged.head()

final shape: (66727, 54)


,OBJECT_NAME,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,NORAD_CAT_ID,BSTAR,MEAN_MOTION_DOT,SEMIMAJOR_AXIS,PERIOD,...,rx_km,ry_km,rz_km,vx_km_s,vy_km_s,vz_km_s,tle_age_days,altitude_km,speed_km_s,regime
0,EXPLORER 1,1970-03-31T00:50:24.429408,16.27546304,0.00247390,33.1468,4,0.00000000000000,0.07718844,6577.283,88.476,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,VANGUARD 1,2026-03-20T17:10:14.647584,10.85959671,0.18369429,34.2414,5,0.00016070833000,0.00000108,8613.759,132.602,...,-8210.998356,-3990.498411,-1923.840662,2.574536,-4.548402,3.446308,1.0,2958.830356,6.260450,MEO
2,SPUTNIK 3,1960-04-04T03:52:47.964864,16.28328133,0.00883180,65.0599,8,0.00000000000000,0.02607090,6575.177,88.434,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,EXPLORER 4,1959-10-02T06:53:14.055072,15.51612549,0.02545340,50.2549,9,0.00000000000000,0.00892850,6790.158,92.806,...,-3519.404089,3015.901726,-4733.148996,-6.414072,-3.952547,2.200901,24277.0,253.542836,7.849007,VLEO
4,VANGUARD 2,2026-03-21T09:09:17.498592,11.90288286,0.14457065,32.8774,11,0.00046155897000,0.00000908,8102.774,120.979,...,-1388.811012,7840.407708,3589.239802,-5.787650,-1.342493,2.607252,0.0,2363.038666,6.488216,MEO


In [24]:
merged.to_parquet(f"{base}/merged/ML_merged.parquet", index=False)
print("saved!!", merged.shape)

saved!! (66727, 54)
